# Azure ETL & Retrieval Pipeline Walkthrough

This notebook documents a local validation run of the retrieval pipeline.

Pipeline stages:
1. Extraction
2. Chunking
3. Embedding generation
4. Indexing
5. Hybrid search

The full local pipeline was executed successfully through `run_local.py`.

## 1. Extraction

The local validation run extracted mixed-format documents from the sample corpus.

Expected extraction logic:
- read PDF manuals
- read Markdown troubleshooting guides
- read TXT policy documents
- preserve metadata such as file type, category, and blob-style path

In [ ]:
from run_local import extract_documents, DOCS_FOLDER

# Example call used in local validation:
# docs = extract_documents(DOCS_FOLDER)

### Local run output

- Extracted 4 documents:
  - docs/manuals/deviceA.pdf
  - docs/manuals/deviceB.pdf
  - docs/policies/security.txt
  - docs/troubleshooting/error101.md

## 2. Chunking

Documents were split into retrieval-friendly text chunks.

In [ ]:
from run_local import chunk_documents

### Local run output

- 6 total chunks produced from 4 documents

## 3. Embedding generation

The local run used the `all-MiniLM-L6-v2` sentence-transformer model to generate embeddings for each chunk.

In [ ]:
from sentence_transformers import SentenceTransformer
from run_local import embed_chunks, MODEL_NAME

# Example call used in local validation:
# model = SentenceTransformer(MODEL_NAME)
# vectors = embed_chunks(chunks, model)

### Local run output

- Embedding matrix shape: `(6, 384)`

## 4. Indexing

The local validation run built a FAISS index over the chunk embeddings.

In [ ]:
from run_local import build_index

# Example call used in local validation:
# index = build_index(vectors)

### Local run output

- FAISS index built with 6 vectors
- Saved artifacts:
  - `faiss_index.bin`
  - `chunks_store.pkl`

## 5. Hybrid search

The local validation run executed hybrid retrieval using:
- vector similarity from FAISS
- keyword hit boosting
- caption generation from chunk text

In [ ]:
from run_local import search

# Example call used in local validation:
# results = search("How do I reset the device to factory settings?", index, chunks, model, top_n=3)

### Example query 1
**Query:** How do I reset the device to factory settings?

Top results from local run:
1. `deviceA` — score 1.883 — category `manuals`
2. `deviceB` — score 1.2858 — category `manuals`
3. `error101` — score 1.0748 — category `troubleshooting`

### Example query 2
**Query:** error code 101 connection timeout fix  
**Filter:** category = troubleshooting

Top results from local run:
1. `error101` — score 1.0843 — category `troubleshooting`
2. `error101` — score 0.7184 — category `troubleshooting`

### Example query 3
**Query:** password and encryption security policy

Top results from local run:
1. `security` — score 1.0194 — category `policies`
2. `deviceA` — score 0.2470 — category `manuals`
3. `security` — score 0.2421 — category `policies`

## Summary

The local validation run completed all five stages successfully:
- extraction
- chunking
- embedding generation
- FAISS indexing
- hybrid search

This notebook documents that workflow and the resulting outputs.